In [23]:
import matplotlib.pyplot as plt
import gymnasium as gym
import numpy as np
import torch
import src.metropolis_hastings as mh

from src.time_varying_vine_copula import tv_vinecop


In [24]:
env = gym.make("InvertedPendulum-v5")  
# Reset environment to start a new episode
observation, info = env.reset(seed=123) 
# observation: what the agent can "see" - cart position, velocity
# info: extra debugging information (usually not needed for basic learning)

print(f"Starting observation: {observation}")

n = 200 #truncation limit
obs_d = env.observation_space.shape[0]
act_d = env.action_space.shape[0]
print(obs_d, act_d)

episode_over = False
total_reward = 0

print(obs_d, act_d)
observations = np.zeros((n+1, obs_d))
actions = np.zeros((n+1, act_d))
set_actions = np.genfromtxt('C:/Users/woodg/Documents/Vine_dissertation_chaeyun/actions.csv', delimiter=',')
set_a_l = len(set_actions)
actions[:set_a_l,0] = set_actions
observations[0] = observation
i=1
while not episode_over:
    # Choose an action: 0 = push cart left, 1 = push cart right
    if i>set_a_l:
        actions[i-1] = env.action_space.sample()  # Random action for now - real agents will be smarter!
    # Take the action and see what happens
    observations[i], reward, terminated, truncated, info = env.step(actions[i-1])

    total_reward += reward
    episode_over = terminated or truncated
    i+=1
if i <= n+1:
    observations = observations[:i]
    actions = actions[:i]
# if i>20:
#     np.savetxt("actions.csv", actions, delimiter=",")
#     exit = True
print(f"Episode finished! Total reward: {total_reward}")
env.close()

data = np.concatenate((observations, actions), axis=1)

print("Starting Fit")
tv = tv_vinecop()
tv.fit(data, check_vines=False)
print("Fit finished")

Starting observation: [ 0.00364704 -0.00892358 -0.0055928  -0.00631256]
4 1
4 1
Episode finished! Total reward: 24
Starting Fit
Fit finished


In [25]:

def target_func(copula:tv_vinecop, action, log = False):
    if log:
        def log_pdf(x):
            return np.log(copula.pdf_predictive(x, action))

        return log_pdf

    def pdf(x):
        return (copula.pdf_predictive(x, action))
            
    return pdf


#print(tv.pdf_predictive(np.array([observations[-2]]),0))
# fig, ax = tv.check_rbp_times_conditional_vine()

# plt.savefig("C:/Users/woodg/Documents/Vine_dissertation_chaeyun/plots_rl/conditionals")
# start = np.repeat([observations[0]], repeats=2, axis = 0)
# start_act = np.repeat([actions[0]], repeats = 2, axis = 0)
# tv.jump_back(start, start_act)

tv.step_forward(np.array([observations[0]]), actions[0][0])

print("Starting Metropolis Hastings")
var = .0001
prop = mh.mvn_def(covariance=var)
sample = mh.sample_mvn(covariance=var)
predicted_next = np.zeros_like(observations)
predicted_std = np.zeros_like(observations)
predicted_next[0] = observations[0]



Starting Metropolis Hastings


In [26]:

for i, act in enumerate(actions[0:-1]):
    target = target_func(tv, act[0], log=True)
    #chain, ar = mh.metropolis_hastings(target, prop, sample, np.array(observations[i+2]), symmetric_proposal=True, chain_length=800, burn_in=500)
    chain = mh.adaptive_mh(target, torch.tensor(predicted_next[i], dtype = torch.double), torch.eye(4, dtype=torch.double)*var, nmoves = 700, return_entire_chain=True, adapt_no=100, burn_in=400)

    # fig, ax = plt.subplots(1,4)
    # for j in range(4):
    #     ax[j].plot(chain[:,j])
    # plt.savefig(f"C:/Users/woodg/Documents/Vine_dissertation_chaeyun/plots_rl/chain{i}")
    # plt.close()
    
    # Calculate prediction, prediction variance
    predicted_next[i+1] = torch.mean(chain, axis = 0)
    predicted_std[i+1] = torch.std(chain, axis = 0)
    print(i, act[0])
    tv.step_forward(np.array([predicted_next[i+1]]), actions[i+1][0])

fig, ax = plt.subplots(2,4)
x = np.arange(len(predicted_next))
#Plot the 
for i in range(4):
    ax[0, i].plot(x[1:], observations[1:,i], label = "Observed")
    ax[0, i].plot(x[1:], predicted_next[1:,i], label = "Prediction")
    ax[0, i].fill_between(
        x[1:],
        predicted_next[1:, i] - predicted_std[1:, i],
        predicted_next[1:, i] + predicted_std[1:, i],
        color = "orange",
        alpha=0.25,
        label="Prediction uncertainty"
    )
    ax[1, i].plot(observations[1:,i], predicted_next[1:,i] - observations[1:,i], '.')

#print(observations[-11], observations[-10], np.mean(chain, axis=0), observations[-9])
plt.savefig("C:/Users/woodg/Documents/Vine_dissertation_chaeyun/plots_rl/roll_out_no_observations")
plt.close()


0.26285714285714284
0 0.6810310482978821
0.30714285714285716
1 -1.2474859952926636
0.25
2 1.5883917808532715
0.2757142857142857
3 0.7587964534759521
0.2914285714285714
4 -1.8273946046829224
0.2957142857142857
5 -0.6533893346786499
0.25857142857142856
6 -2.4819536209106445
0.3914285714285714
7 0.00031720998231321573
0.18571428571428572
8 1.0118399858474731
0.2757142857142857
9 2.0199968814849854
0.2785714285714286
10 0.5050217509269714
0.29428571428571426
11 -0.7968438267707825
0.33285714285714285
12 2.14612078666687
0.2642857142857143
13 0.6605178117752075
0.33285714285714285
14 1.169275164604187
0.3271428571428571
15 -2.944810152053833
0.4157142857142857
16 -0.8712737560272217
0.30142857142857143
17 -2.0167198181152344
0.5614285714285714
18 0.39869874715805054
0.38857142857142857
19 0.13447576761245728
0.31
20 2.401761770248413
0.3457142857142857
21 -2.8934836387634277
0.2957142857142857
22 -2.8224680423736572
0.29285714285714287
23 1.4855791330337524
0.2857142857142857
24 2.138761758

In [27]:
print(predicted_std)

[[0.         0.         0.         0.        ]
 [0.00421792 0.00272783 0.12059744 0.18742461]
 [0.01152742 0.00243429 0.13987581 0.38206661]
 [0.01375868 0.0070543  0.08992621 0.15471352]
 [0.00812387 0.0197125  0.06870233 0.29687365]
 [0.00424634 0.00776327 0.12470847 0.06778643]
 [0.02475676 0.008586   0.16686412 0.12538124]
 [0.03678577 0.00077205 0.12868508 0.28487657]
 [0.01002356 0.00821432 0.0060031  0.0096705 ]
 [0.00355481 0.01084622 0.05802899 0.09983921]
 [0.00233891 0.02647799 0.11519352 0.11500887]
 [0.00617844 0.0177462  0.07327609 0.16102681]
 [0.0130426  0.00826327 0.12300541 0.17440915]
 [0.01136147 0.02709661 0.10988811 0.29309536]
 [0.00365658 0.00123402 0.05137813 0.07990842]
 [0.0184883  0.02529413 0.09745074 0.18734515]
 [0.01491307 0.00968633 0.11169546 0.17984903]
 [0.00168386 0.00131275 0.16262333 0.12349908]
 [0.0049494  0.0095871  0.14389314 0.22661874]
 [0.00123784 0.00107606 0.00419268 0.01897918]
 [0.01535141 0.00600943 0.0987454  0.20310144]
 [0.01164437 